# Customer Behavior Analysis using Big Data Technologies

## Project Overview

This project analyzes customer behavior in an e-commerce environment using a combination of Big Data tools and database technologies.

The system integrates multiple datasets including customer information, orders, products and customer interaction events.

Technologies used:

- Apache Spark (Databricks) for data integration and transformation
- JSON document data for event storage
- Spark SQL for data analysis
- Neo4j Graph Database for relationship analysis

The goal is to understand customer purchasing behavior and product relationships.

## Data Sources

The project uses multiple datasets representing an e-commerce platform.

Datasets used:

- **customers.csv** – customer information
- **orders.csv** – purchase transactions
- **products.csv** – product catalog
- **events.csv** – customer interaction events

These datasets simulate an online retail system where customers interact with products and place orders.

In [0]:
customers = spark.read.csv("/Volumes/workspace/default/project_data/customers.csv", header=True, inferSchema=True)

orders = spark.read.csv("/Volumes/workspace/default/project_data/orders.csv", header=True, inferSchema=True)

events = spark.read.csv("/Volumes/workspace/default/project_data/events.csv", header=True, inferSchema=True)

products = spark.read.csv("/Volumes/workspace/default/project_data/FashionDataset (1).csv", header=True, inferSchema=True)

## Data Exploration

Before processing the data, we inspect the datasets to understand their structure and verify that they were loaded correctly.

The first five rows of each dataset are displayed using the `.limit()` function.

In [0]:
display(customers.limit(5))
display(orders.limit(5))
display(events.limit(5))
display(products.limit(5))

customer_id,name,age,city,email,registered_date
1,Marie Pettersson,42,Halmstad,zjonsson@example.org,2022-02-08
2,Daniel Olofsson,32,Piteå,signeaberg@example.com,2024-07-26
3,Oskar Hanna,64,Borlänge,erika58@example.net,2022-04-11
4,Eva Östman,61,Örebro,zpersson@example.com,2024-08-20
5,Carlos Lindström,65,Södertälje,whansson@example.com,2022-04-15


order_id,customer_id,product_id,quantity,order_date,total_price,status
1,757,26754,2,2025-05-15,1410,completed
2,338,3083,1,2026-02-07,689,cancelled
3,905,16145,1,2024-08-02,573,returned
4,387,17909,3,2025-09-16,1650,cancelled
5,160,7541,3,2024-05-05,2400,returned


event_id,customer_id,product_id,event_type,timestamp,device,location
1,534,5601,purchase,2025-07-28T02:09:41.000Z,desktop,Stephanieland
2,342,2140,purchase,2025-08-25T04:51:00.000Z,tablet,Stevefort
3,654,16687,wishlist,2024-08-16T04:45:36.000Z,desktop,Shermanton
4,515,29522,remove_from_cart,2024-08-02T03:30:17.000Z,desktop,Ronaldview
5,793,22682,review,2024-06-13T20:57:09.000Z,mobile,Lake Carriemouth


_c0,BrandName,Deatils,Sizes,MRP,SellPrice,Discount,Category
0,life,solid cotton blend collar neck womens a-line dress - indigo,"Size:Large,Medium,Small,X-Large,X-Small",Rs,null,null,null
"1699""",849,50% off,Westernwear-Women,null,null,null,null
1,only,polyester peter pan collar womens blouson dress - yellow,"Size:34,36,38,40",Rs,null,null,null
"3499""",2449,30% off,Westernwear-Women,null,null,null,null
2,fratini,solid polyester blend wide neck womens regular top - off white,"Size:Large,X-Large,XX-Large",Rs,null,null,null


In [0]:
print(customers.count(), orders.count(), events.count(), products.count())

15000 20000 15000 53308


## Data Integration with Apache Spark

Apache Spark is used to integrate multiple datasets into a unified processing environment.

The data is loaded into Spark DataFrames and registered as temporary SQL views.  
This allows us to query and analyze the data using Spark SQL.

In [0]:
customers.createOrReplaceTempView("customers_raw")
orders.createOrReplaceTempView("orders_raw")
events.createOrReplaceTempView("events_raw")
products.createOrReplaceTempView("products_raw")

In [0]:
spark.sql("SELECT COUNT(*) AS n_customers FROM customers_raw").show()
spark.sql("SELECT COUNT(*) AS n_orders FROM orders_raw").show()
spark.sql("SELECT COUNT(*) AS n_events FROM events_raw").show()
spark.sql("SELECT COUNT(*) AS n_products FROM products_raw").show()

+-----------+
|n_customers|
+-----------+
|      15000|
+-----------+

+--------+
|n_orders|
+--------+
|   20000|
+--------+

+--------+
|n_events|
+--------+
|   15000|
+--------+

+----------+
|n_products|
+----------+
|     53308|
+----------+



## Document Data (JSON)

Customer interaction data was converted into JSON format to simulate document database storage.

JSON is useful for storing semi-structured event data such as user behavior, product views, and interactions.

This structure is similar to document databases such as MongoDB.

In [0]:
events.write.mode("overwrite").json("/Volumes/workspace/default/project_data/events_json")
events_json = spark.read.json("/Volumes/workspace/default/project_data/events_json")
events_json.createOrReplaceTempView("events_json")
display(events_json.limit(5))

customer_id,device,event_id,event_type,location,product_id,timestamp
534,desktop,1,purchase,Stephanieland,5601,2025-07-28T02:09:41.000Z
342,tablet,2,purchase,Stevefort,2140,2025-08-25T04:51:00.000Z
654,desktop,3,wishlist,Shermanton,16687,2024-08-16T04:45:36.000Z
515,desktop,4,remove_from_cart,Ronaldview,29522,2024-08-02T03:30:17.000Z
793,mobile,5,review,Lake Carriemouth,22682,2024-06-13T20:57:09.000Z


## Data Transformation

The datasets were cleaned and transformed using Apache Spark.

Common transformations include:

- removing duplicates
- trimming string values
- converting data types
- preparing data for analysis

These transformations ensure the data is consistent and usable for further analysis.

In [0]:
from pyspark.sql.functions import col, to_timestamp, trim

customers_s = customers.dropDuplicates()
orders_s = orders.dropDuplicates()
products_s = products.dropDuplicates()
events_s = events_json.dropDuplicates()

# If your events has a time column, try parsing it (adjust name if needed)
# Example: events_s = events_s.withColumn("timestamp", to_timestamp(col("timestamp")))

customers_s.createOrReplaceTempView("customers")
orders_s.createOrReplaceTempView("orders")
products_s.createOrReplaceTempView("products")
events_s.createOrReplaceTempView("events")

## Data Analysis

Spark SQL queries are used to analyze customer purchasing behavior and product interactions.

Examples of analysis performed include:

- identifying the most popular products
- understanding customer interaction patterns
- analyzing order relationships between customers and products

In [0]:
%sql
SELECT event_type, COUNT(*) AS total
FROM events
GROUP BY event_type
ORDER BY total DESC;

event_type,total
remove_from_cart,2563
add_to_cart,2562
review,2517
view_product,2461
purchase,2460
wishlist,2437


In [0]:
%sql
SELECT customer_id, COUNT(*) AS orders_count
FROM orders
GROUP BY customer_id
ORDER BY orders_count DESC
LIMIT 10;

customer_id,orders_count
184,39
325,35
799,34
85,33
430,32
958,32
920,32
296,31
642,31
789,31


In [0]:
%sql
SELECT product_id, COUNT(*) AS times_ordered
FROM orders
GROUP BY product_id
ORDER BY times_ordered DESC
LIMIT 10;

product_id,times_ordered
5665,5
22206,5
19958,5
1697,5
3886,5
15102,5
29009,5
30447,5
28954,5
27953,5


In [0]:
from pyspark.sql.functions import col, regexp_replace, trim

products_clean = (products
    .withColumn("_c0_clean", trim(regexp_replace(col("_c0"), '[^0-9]', '')))
    .withColumn("product_id", col("_c0_clean").cast("long"))
    .drop("_c0_clean")
)

products_clean.createOrReplaceTempView("products_clean")
orders.createOrReplaceTempView("orders")

In [0]:
%sql
SELECT event_type, COUNT(*) AS total
FROM events_json
GROUP BY event_type
ORDER BY total DESC;

event_type,total
remove_from_cart,2563
add_to_cart,2562
review,2517
view_product,2461
purchase,2460
wishlist,2437


In [0]:
%sql
SELECT customer_id, SUM(total_price) AS total_spent
FROM orders
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10;

customer_id,total_spent
184,43101
814,35686
85,34822
296,34337
539,34015
817,33373
974,33312
788,33240
728,32621
430,32518


In [0]:
%sql
SELECT product_id, COUNT(*) AS times_ordered
FROM orders
GROUP BY product_id
ORDER BY times_ordered DESC
LIMIT 10;

product_id,times_ordered
30447,5
22206,5
23605,5
24451,5
19958,5
23308,5
27303,5
18811,5
1697,5
27953,5


In [0]:
%sql
SELECT p.BrandName, COUNT(*) AS sales
FROM orders o
JOIN products_clean p
ON o.product_id = p.product_id
GROUP BY p.BrandName
ORDER BY sales DESC
LIMIT 10;

BrandName,sales
749,842
vastranand,618
599,598
499,592
649,479
349,466
399,455
249,440
2159,404
stop,391


In [0]:
%sql
SELECT p.Category, COUNT(*) AS sales
FROM orders o
JOIN products_clean p
ON o.product_id = p.product_id
GROUP BY p.Category
ORDER BY sales DESC
LIMIT 10;

Category,sales
null,25596
Westernwear-Women,686
Indianwear-Women,686
Lingerie&Nightwear-Women,476
Footwear-Women,331
Watches-Women,286
Jewellery-Women,260
Fragrance-Women,126


In [0]:
%sql
WITH views AS (
  SELECT customer_id, COUNT(*) AS views
  FROM events_json
  WHERE event_type = 'view'
  GROUP BY customer_id
),
purchases AS (
  SELECT customer_id, COUNT(*) AS purchases
  FROM events_json
  WHERE event_type = 'purchase'
  GROUP BY customer_id
)
SELECT v.customer_id, v.views, COALESCE(p.purchases, 0) AS purchases
FROM views v
LEFT JOIN purchases p ON v.customer_id = p.customer_id
ORDER BY v.views DESC
LIMIT 20;

customer_id,views,purchases


## Gold Layer Results

The final Spark SQL results are saved as Delta tables in the Gold layer.

These tables contain business-ready outputs that can be reused for reporting and presentation:

- `gold_top_brands`
- `gold_top_categories`


In [0]:
spark.sql("""
SELECT p.BrandName, COUNT(*) AS sales
FROM orders o
JOIN products_clean p
ON o.product_id = p.product_id
GROUP BY p.BrandName
ORDER BY sales DESC
LIMIT 10
""").write.mode("overwrite").format("delta").saveAsTable("gold_top_brands")

spark.sql("""
SELECT p.Category, COUNT(*) AS sales
FROM orders o
JOIN products_clean p
ON o.product_id = p.product_id
GROUP BY p.Category
ORDER BY sales DESC
LIMIT 10
""").write.mode("overwrite").format("delta").saveAsTable("gold_top_categories")

In [0]:
%sql
SELECT * FROM gold_top_brands;

BrandName,sales
749,842
vastranand,618
599,598
499,592
649,479
349,466
399,455
249,440
2159,404
stop,391


## Graph Data Export

The transformed datasets were exported as CSV files to be used in Neo4j for graph analysis.

The export includes:

- customer nodes
- product nodes
- customer-product relationships
- customer-order relationships
- order-product relationships


In [0]:
# Nodes
customers.select("customer_id").dropDuplicates() \
  .write.mode("overwrite").option("header", True) \
  .csv("/Volumes/workspace/default/project_data/neo4j/nodes_customers")

products_clean.select("product_id", "BrandName", "Category").dropDuplicates() \
  .write.mode("overwrite").option("header", True) \
  .csv("/Volumes/workspace/default/project_data/neo4j/nodes_products")

# Relationships: customer -> product (from events)
events_json.select("customer_id", "product_id", "event_type") \
  .dropna().dropDuplicates() \
  .write.mode("overwrite").option("header", True) \
  .csv("/Volumes/workspace/default/project_data/neo4j/rel_customer_product")

# Relationships: customer -> order and order -> product (from orders)
orders.select("order_id", "customer_id").dropna().dropDuplicates() \
  .write.mode("overwrite").option("header", True) \
  .csv("/Volumes/workspace/default/project_data/neo4j/rel_customer_order")

orders.select("order_id", "product_id").dropna().dropDuplicates() \
  .write.mode("overwrite").option("header", True) \
  .csv("/Volumes/workspace/default/project_data/neo4j/rel_order_product")

## Conclusion

This project demonstrated how Apache Spark can be used to integrate and transform multiple datasets in a big data workflow.

JSON was used to represent semi-structured event data, while Neo4j was used to model relationships between customers, orders and products.

The final solution supports both SQL-based analysis and graph-based relationship analysis.
